In [51]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

class ImprovedSpaceshipTitanicModel:
    def __init__(self):
        self.numeric_features = None
        self.categorical_features = None
        self.model = None
        self.preprocessor = None
        self.le_dict = {}

    def extract_title(self, name):
        """Extract title from passenger name with more granular categories"""
        if pd.isna(name):
            return 'Unknown'
        try:
            title = name.split(',')[1].split('.')[0].strip()
            # Group similar titles
            title_mapping = {
                'Mr': 'Mr',
                'Mrs': 'Mrs',
                'Miss': 'Miss',
                'Ms': 'Miss',
                'Dr': 'Officer',
                'Col': 'Officer',
                'Major': 'Officer',
                'Lady': 'Royalty',
                'Sir': 'Royalty',
                'Countess': 'Royalty',
                'Don': 'Royalty',
                'Dona': 'Royalty',
                'Rev': 'Religious',
                'Master': 'Master'
            }
            return title_mapping.get(title, 'Other')
        except:
            return 'Unknown'

    def extract_name_features(self, df):
        """Extract advanced name-based features"""
        df['Name_Length'] = df['Name'].fillna('').str.len()
        df['Name_Words'] = df['Name'].fillna('').str.count(' ') + 1
        df['Has_Multiple_Titles'] = df['Name'].fillna('').str.count('\\.').gt(1).astype(int)
        return df

    def process_cabin(self, df):
        """Process cabin information"""
        df['Cabin'] = df['Cabin'].fillna('X/0/X')
        cabin_split = df['Cabin'].str.split('/', expand=True)
        df['Deck'] = cabin_split[0]
        df['Cabin_Num'] = pd.to_numeric(cabin_split[1], errors='coerce')
        df['Side'] = cabin_split[2]

        # Fill missing values
        df['Deck'] = df['Deck'].replace('X', 'Unknown')
        df['Side'] = df['Side'].replace('X', 'Unknown')
        df['Cabin_Num'] = df['Cabin_Num'].fillna(df['Cabin_Num'].median())

        return df

    def enhance_features(self, df):
        """Create advanced features with more sophisticated engineering"""
        df = df.copy()

        # Spending features with log transformation
        spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
        df['TotalSpending'] = df[spending_cols].sum(axis=1)
        df['LogTotalSpending'] = np.log1p(df['TotalSpending'])

        # Spending patterns
        df['SpendingVariety'] = (df[spending_cols] > 0).sum(axis=1)
        df['MaxSpendingService'] = df[spending_cols].idxmax(axis=1)
        df['SpendingConcentration'] = df[spending_cols].max(axis=1) / (df['TotalSpending'] + 1)

        # Advanced age interactions
        df['Age_Squared'] = df['Age'] ** 2
        df['Age_Decade'] = (df['Age'] // 10) * 10

        # Cabin features
        df['IsPremiumDeck'] = df['Deck'].isin(['A', 'B']).astype(int)
        df['IsEconomyDeck'] = df['Deck'].isin(['F', 'G']).astype(int)

        # Create deck amenity scores
        deck_amenity_scores = {
            'A': 5, 'B': 4, 'C': 3, 'D': 2, 'E': 2, 'F': 1, 'G': 1, 'Unknown': 0
        }
        df['DeckAmenityScore'] = df['Deck'].map(deck_amenity_scores)

        # Group features
        df['GroupSpendingRank'] = df.groupby('GroupId')['TotalSpending'].rank(pct=True)
        df['GroupAgeRank'] = df.groupby('GroupId')['Age'].rank(pct=True)

        # Interaction features
        df['Spending_Per_Person'] = df['TotalSpending'] / df['GroupSize']
        df['IsVIP_HighSpender'] = ((df['VIP'] == 1) & (df['TotalSpending'] > df['TotalSpending'].median())).astype(int)

        # Create advanced categorical combinations
        df['Destination_Class'] = df['Destination'] + '_' + df['Deck'].astype(str)
        df['Home_Destination'] = df['HomePlanet'] + '_' + df['Destination']

        # Time-based features from passenger ID
        df['PassengerNumber'] = df['PassengerId'].str.split('_').str[1].astype(int)
        df['IsEarlyBooking'] = (df['PassengerNumber'] <= df['PassengerNumber'].median()).astype(int)

        return df

    def preprocess_data(self, df):
        """Enhanced preprocessing with sophisticated handling of missing values and feature engineering"""
        df = df.copy()

        # Handle missing names
        df['Name'] = df['Name'].fillna('Unknown Unknown')

        # Extract name features
        df = self.extract_name_features(df)
        df['Title'] = df['Name'].apply(self.extract_title)

        # Process Cabin first
        df = self.process_cabin(df)

        # Extract GroupId and create group features
        df['GroupId'] = df['PassengerId'].str.split('_').str[0]
        df['GroupSize'] = df.groupby('GroupId')['PassengerId'].transform('count')

        # Handle missing values in numeric columns using KNN imputation
        numeric_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
        knn_imputer = KNNImputer(n_neighbors=5)
        df[numeric_cols] = pd.DataFrame(
            knn_imputer.fit_transform(df[numeric_cols]),
            columns=numeric_cols,
            index=df.index
        )

        # Categorical imputation using most frequent value within group
        categorical_cols = ['HomePlanet', 'Destination']
        for col in categorical_cols:
            df[col] = df.groupby('Title')[col].transform(
                lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else 'Unknown')
            )

        # Convert boolean columns
        bool_cols = ['CryoSleep', 'VIP']
        for col in bool_cols:
            df[col] = df[col].map({'True': 1, 'False': 0}).fillna(0).astype(int)

        # Apply enhanced features
        df = self.enhance_features(df)

        return df

    def setup_pipeline(self):
        """Set up the preprocessing pipeline with additional features"""
        self.numeric_features = [
            'Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
            'TotalSpending', 'LogTotalSpending', 'SpendingVariety', 'SpendingConcentration',
            'Age_Squared', 'Age_Decade', 'DeckAmenityScore', 'GroupSpendingRank',
            'GroupAgeRank', 'Spending_Per_Person', 'PassengerNumber', 'Name_Length',
            'Name_Words', 'Has_Multiple_Titles', 'GroupSize', 'Cabin_Num'
        ]

        self.categorical_features = [
            'HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side',
            'Title', 'MaxSpendingService', 'IsPremiumDeck', 'IsEconomyDeck',
            'IsVIP_HighSpender', 'IsEarlyBooking', 'Destination_Class', 'Home_Destination'
        ]

        numeric_transformer = Pipeline([
            ('imputer', KNNImputer(n_neighbors=5)),
            ('scaler', StandardScaler())
        ])

        categorical_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])

        self.preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, self.numeric_features),
                ('cat', categorical_transformer, self.categorical_features)
            ])

    def create_model(self):
        """Create an XGBoost model with optimized hyperparameters"""
        return xgb.XGBClassifier(
            learning_rate=0.01,
            n_estimators=2000,
            max_depth=6,
            min_child_weight=1,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=1,
            reg_alpha=0.1,
            reg_lambda=1,
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss'
        )

    def train_and_predict(self, train_path, test_path):
        """Train the model with validation split and make predictions"""
        print("Loading and preprocessing data...")
        train_data = pd.read_csv(train_path)
        test_data = pd.read_csv(test_path)

        train_data = self.preprocess_data(train_data)
        test_data = self.preprocess_data(test_data)

        print("Setting up pipeline...")
        self.setup_pipeline()

        X = train_data[self.numeric_features + self.categorical_features]
        y = train_data['Transported'].fillna(False)
        y = y.map({True: 1, False: 0, 'True': 1, 'False': 0})

        # Split data for validation
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

        # Initialize model for validation phase
        print("Training XGBoost model...")
        validation_model = xgb.XGBClassifier(
            learning_rate=0.01,
            n_estimators=2000,
            max_depth=6,
            min_child_weight=1,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=1,
            reg_alpha=0.1,
            reg_lambda=1,
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss',
            early_stopping_rounds=50
        )

        # Prepare preprocessed data
        print("Preprocessing training data...")
        preprocessed_X_train = self.preprocessor.fit_transform(X_train)
        preprocessed_X_val = self.preprocessor.transform(X_val)

        # Train model with validation
        print("Fitting model with validation...")
        eval_set = [(preprocessed_X_train, y_train), (preprocessed_X_val, y_val)]

        validation_model.fit(
            preprocessed_X_train,
            y_train,
            eval_set=eval_set,
            verbose=100
        )

        # Get optimal number of rounds from validation
        best_iteration = validation_model.best_iteration

        # Evaluate on validation set
        val_pred = validation_model.predict(preprocessed_X_val)
        val_accuracy = accuracy_score(y_val, val_pred)
        print(f"\nValidation accuracy: {val_accuracy:.4f}")
        print("\nClassification Report:")
        print(classification_report(y_val, val_pred))

        # Initialize final model with optimal iterations
        print("\nRetraining on full dataset...")
        self.model = xgb.XGBClassifier(
            learning_rate=0.01,
            n_estimators=best_iteration,
            max_depth=6,
            min_child_weight=1,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=1,
            reg_alpha=0.1,
            reg_lambda=1,
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss'
        )

        # Train final model on full dataset
        preprocessed_X_full = self.preprocessor.fit_transform(X)
        self.model.fit(preprocessed_X_full, y, verbose=100)

        # Make predictions on test data
        print("Making predictions on test data...")
        preprocessed_X_test = self.preprocessor.transform(test_data[self.numeric_features + self.categorical_features])
        test_pred = self.model.predict(preprocessed_X_test)

        # Create submission
        submission_df = pd.DataFrame({
            'PassengerId': test_data['PassengerId'],
            'Transported': test_pred.astype(bool)
        })

        submission_df.to_csv('submission.csv', index=False)
        print("\nSubmission file created successfully!")

        return submission_df

    def extract_title(self, name):
        """Extract title from passenger name with more granular categories"""
        if pd.isna(name):
            return 'Unknown'
        try:
            title = name.split(',')[1].split('.')[0].strip()
            # Group similar titles
            title_mapping = {
                'Mr': 'Mr',
                'Mrs': 'Mrs',
                'Miss': 'Miss',
                'Ms': 'Miss',
                'Dr': 'Officer',
                'Col': 'Officer',
                'Major': 'Officer',
                'Lady': 'Royalty',
                'Sir': 'Royalty',
                'Countess': 'Royalty',
                'Don': 'Royalty',
                'Dona': 'Royalty',
                'Rev': 'Religious',
                'Master': 'Master'
            }
            return title_mapping.get(title, 'Other')
        except:
            return 'Unknown'

    def extract_name_features(self, df):
        """Extract advanced name-based features"""
        df['Name_Length'] = df['Name'].fillna('').str.len()
        df['Name_Words'] = df['Name'].fillna('').str.count(' ') + 1
        df['Has_Multiple_Titles'] = df['Name'].fillna('').str.count('\\.').gt(1).astype(int)
        return df

    def process_cabin(self, df):
        """Process cabin information"""
        df['Cabin'] = df['Cabin'].fillna('X/0/X')
        cabin_split = df['Cabin'].str.split('/', expand=True)
        df['Deck'] = cabin_split[0]
        df['Cabin_Num'] = pd.to_numeric(cabin_split[1], errors='coerce')
        df['Side'] = cabin_split[2]

        # Fill missing values
        df['Deck'] = df['Deck'].replace('X', 'Unknown')
        df['Side'] = df['Side'].replace('X', 'Unknown')
        df['Cabin_Num'] = df['Cabin_Num'].fillna(df['Cabin_Num'].median())

        return df

    def enhance_features(self, df):
        """Create advanced features with more sophisticated engineering"""
        df = df.copy()

        # Spending features with log transformation
        spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
        df['TotalSpending'] = df[spending_cols].sum(axis=1)
        df['LogTotalSpending'] = np.log1p(df['TotalSpending'])

        # Spending patterns
        df['SpendingVariety'] = (df[spending_cols] > 0).sum(axis=1)
        df['MaxSpendingService'] = df[spending_cols].idxmax(axis=1)
        df['SpendingConcentration'] = df[spending_cols].max(axis=1) / (df['TotalSpending'] + 1)

        # Advanced age interactions
        df['Age_Squared'] = df['Age'] ** 2
        df['Age_Decade'] = (df['Age'] // 10) * 10

        # Cabin features
        df['IsPremiumDeck'] = df['Deck'].isin(['A', 'B']).astype(int)
        df['IsEconomyDeck'] = df['Deck'].isin(['F', 'G']).astype(int)

        # Create deck amenity scores
        deck_amenity_scores = {
            'A': 5, 'B': 4, 'C': 3, 'D': 2, 'E': 2, 'F': 1, 'G': 1, 'Unknown': 0
        }
        df['DeckAmenityScore'] = df['Deck'].map(deck_amenity_scores)

        # Group features
        df['GroupSpendingRank'] = df.groupby('GroupId')['TotalSpending'].rank(pct=True)
        df['GroupAgeRank'] = df.groupby('GroupId')['Age'].rank(pct=True)

        # Interaction features
        df['Spending_Per_Person'] = df['TotalSpending'] / df['GroupSize']
        df['IsVIP_HighSpender'] = ((df['VIP'] == 1) & (df['TotalSpending'] > df['TotalSpending'].median())).astype(int)

        # Create advanced categorical combinations
        df['Destination_Class'] = df['Destination'] + '_' + df['Deck'].astype(str)
        df['Home_Destination'] = df['HomePlanet'] + '_' + df['Destination']

        # Time-based features from passenger ID
        df['PassengerNumber'] = df['PassengerId'].str.split('_').str[1].astype(int)
        df['IsEarlyBooking'] = (df['PassengerNumber'] <= df['PassengerNumber'].median()).astype(int)

        return df

    def preprocess_data(self, df):
        """Enhanced preprocessing with sophisticated handling of missing values and feature engineering"""
        df = df.copy()

        # Handle missing names
        df['Name'] = df['Name'].fillna('Unknown Unknown')

        # Extract name features
        df = self.extract_name_features(df)
        df['Title'] = df['Name'].apply(self.extract_title)

        # Process Cabin first
        df = self.process_cabin(df)

        # Extract GroupId and create group features
        df['GroupId'] = df['PassengerId'].str.split('_').str[0]
        df['GroupSize'] = df.groupby('GroupId')['PassengerId'].transform('count')

        # Handle missing values in numeric columns using KNN imputation
        numeric_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
        knn_imputer = KNNImputer(n_neighbors=5)
        df[numeric_cols] = pd.DataFrame(
            knn_imputer.fit_transform(df[numeric_cols]),
            columns=numeric_cols,
            index=df.index
        )

        # Categorical imputation using most frequent value within group
        categorical_cols = ['HomePlanet', 'Destination']
        for col in categorical_cols:
            df[col] = df.groupby('Title')[col].transform(
                lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else 'Unknown')
            )

        # Convert boolean columns
        bool_cols = ['CryoSleep', 'VIP']
        for col in bool_cols:
            df[col] = df[col].map({'True': 1, 'False': 0}).fillna(0).astype(int)

        # Apply enhanced features
        df = self.enhance_features(df)

        return df

    def setup_pipeline(self):
        """Set up the preprocessing pipeline with additional features"""
        self.numeric_features = [
            'Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
            'TotalSpending', 'LogTotalSpending', 'SpendingVariety', 'SpendingConcentration',
            'Age_Squared', 'Age_Decade', 'DeckAmenityScore', 'GroupSpendingRank',
            'GroupAgeRank', 'Spending_Per_Person', 'PassengerNumber', 'Name_Length',
            'Name_Words', 'Has_Multiple_Titles', 'GroupSize', 'Cabin_Num'
        ]

        self.categorical_features = [
            'HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side',
            'Title', 'MaxSpendingService', 'IsPremiumDeck', 'IsEconomyDeck',
            'IsVIP_HighSpender', 'IsEarlyBooking', 'Destination_Class', 'Home_Destination'
        ]

        numeric_transformer = Pipeline([
            ('imputer', KNNImputer(n_neighbors=5)),
            ('scaler', StandardScaler())
        ])

        categorical_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])

        self.preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, self.numeric_features),
                ('cat', categorical_transformer, self.categorical_features)
            ])

In [52]:
model = ImprovedSpaceshipTitanicModel()
submission = model.train_and_predict('train.csv', 'test.csv')

Loading and preprocessing data...
Setting up pipeline...
Training XGBoost model...
Preprocessing training data...
Fitting model with validation...
[0]	validation_0-logloss:0.68845	validation_1-logloss:0.68870
[100]	validation_0-logloss:0.44897	validation_1-logloss:0.48085
[200]	validation_0-logloss:0.37755	validation_1-logloss:0.43152
[300]	validation_0-logloss:0.34361	validation_1-logloss:0.41319
[400]	validation_0-logloss:0.32419	validation_1-logloss:0.40636
[500]	validation_0-logloss:0.31066	validation_1-logloss:0.40306
[600]	validation_0-logloss:0.29977	validation_1-logloss:0.40108
[700]	validation_0-logloss:0.28899	validation_1-logloss:0.39972
[800]	validation_0-logloss:0.27918	validation_1-logloss:0.39894
[835]	validation_0-logloss:0.27653	validation_1-logloss:0.39898

Validation accuracy: 0.8016

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.79      0.80       861
           1       0.80      0.81      0.80       878